# Custom Feature Engineering Transformer

In [3]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

class CreditRiskFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom scikit-learn transformer to cap outliers and engineer new features.
    """
    def __init__(self):
        # We can pass hyperparameters here if needed (e.g., capping thresholds)
        pass
        
    def fit(self, X, y=None):
        # fit() is used to learn statistics from the training data (like mean or max).
        # Since our logic relies on hardcoded business rules (not learned stats), we just return self.
        return self
        
    def transform(self, X):
        # Always operate on a copy to avoid SettingWithCopy warnings and unintended mutations
        X_out = X.copy()
        
        # 1. Cap Revolving Utilization (Business rule: utilization rarely exceeds 200% or 2.0)
        if 'RevolvingUtilizationOfUnsecuredLines' in X_out.columns:
            X_out['RevolvingUtilizationOfUnsecuredLines'] = np.where(
                X_out['RevolvingUtilizationOfUnsecuredLines'] > 2.0, 
                2.0, 
                X_out['RevolvingUtilizationOfUnsecuredLines']
            )
            
        # 2. Cap Age (Data entry errors: replace age 0 with the dataset median or a logical minimum)
        if 'age' in X_out.columns:
            X_out['age'] = np.where(X_out['age'] < 18, 18, X_out['age'])
            
        # 3. Engineer Composite Feature: Total Past Due Events
        past_due_cols = [
            'NumberOfTime30-59DaysPastDueNotWorse', 
            'NumberOfTime60-89DaysPastDueNotWorse', 
            'NumberOfTimes90DaysLate'
        ]
        if all(col in X_out.columns for col in past_due_cols):
            X_out['TotalPastDue'] = X_out[past_due_cols].sum(axis=1)
            
        return X_out

print("Custom transformer 'CreditRiskFeatureEngineer' initialized successfully.")

Custom transformer 'CreditRiskFeatureEngineer' initialized successfully.


# Constructing the Preprocessing Pipeline

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Define the sequence of transformations
preprocessing_pipeline = Pipeline(steps=[
    # Step 1: Execute our custom business logic and feature engineering
    ('feature_engineer', CreditRiskFeatureEngineer()),
    
    # Step 2: Fill missing values (Income and Dependents) with the median
    ('imputer', SimpleImputer(strategy='median')),
    
    # Step 3: Scale all features so they have a mean of 0 and variance of 1
    ('scaler', StandardScaler())
])

print("Preprocessing pipeline constructed.")

Preprocessing pipeline constructed.


# Splitting and Applying the Pipeline

In [5]:
# Load the raw data
df = pd.read_csv('../data/cs-training.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

# 1. Split the data FIRST
# stratify=y ensures the 93/7 class imbalance is maintained in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Fit and Transform the TRAINING data
# The pipeline learns the medians and scaling parameters only from X_train
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# 3. ONLY Transform the TEST data
# The pipeline applies the medians and scaling parameters learned from X_train to X_test
X_test_processed = preprocessing_pipeline.transform(X_test)

# Convert back to DataFrame for inspection (the pipeline outputs NumPy arrays)
# Note: SimpleImputer and StandardScaler drop column names, so we must reassign them
final_columns = list(X.columns) + ['TotalPastDue'] # Add our engineered feature name
X_train_df = pd.DataFrame(X_train_processed, columns=final_columns)

print(f"Original X_train shape: {X_train.shape}")
print(f"Processed X_train shape: {X_train_processed.shape}")
display(X_train_df.head())

Original X_train shape: (120000, 10)
Processed X_train shape: (120000, 11)


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,TotalPastDue
0,-0.572983,0.657515,-0.100445,0.711051,-0.077669,-0.671428,-0.063715,-0.016033,-0.057873,1.140084,-0.074383
1,-0.864640,1.402229,-0.100445,-0.168015,-0.199624,-0.477692,-0.063715,-0.016033,-0.057873,-0.666121,-0.074383
2,-0.299896,-1.373523,-0.100445,-0.168152,-0.206067,-0.090221,-0.063715,-0.898607,-0.057873,1.140084,-0.074383
3,1.855666,0.522113,-0.100445,-0.111893,-0.077669,-0.671428,-0.063715,-0.898607,-0.057873,-0.666121,-0.074383
4,-0.256012,0.522113,-0.100445,-0.167767,-0.260985,0.297250,-0.063715,-0.016033,-0.057873,-0.666121,-0.074383
